# Notebook 05: Historical Trends & Center Trajectories
## TransPlan Validation & Reproducibility Pack (Phase 4 M3)

This notebook validates the **historical trends analysis** introduced in Phase 4 Milestone 3. While earlier notebooks focus on point-in-time snapshots (current wait times, current survival rates), this one asks: **how are transplant centers changing over time?**

We cover:

1. **Data overview** — multi-year SRTR data (2019-2025) for 22 cities and 6 organs
2. **National trends** — median wait time and graft survival trajectories for all organs
3. **City-level trend heatmap** — kidney wait-time slope across all 22 cities
4. **Trend direction distribution** — improving/stable/declining counts per organ
5. **Sparkline gallery** — kidney wait-time sparklines for all 22 cities vs. national overlay
6. **Regression quality** — R-squared values for linear trend model (kidney)
7. **COVID-19 impact** — 2020 volume dip and recovery pattern across organs
8. **Summary and limitations**

### Data Source
Synthetic seed data derived from SRTR Program-Specific Reports (PSR), 2019-2025. File: `data/srtr-historical.json`. Structure: per city/organ arrays of yearly metrics (median_wait_months, volume, graft_survival_1yr, mortality_rate, delisting_rate, patient_survival_1yr, wait_time_factor).

National aggregate data also available at `data["national"][organ]` with years, median_wait_months, and graft_survival_1yr.

### Trend Computation
The backend service (`services/trends.py`) computes:
- Linear regression (`scipy.stats.linregress`) per metric time series
- Direction classification: improving/stable/declining based on p < 0.10 AND minimum annual change thresholds
- Overall direction via weighted voting (wait_time:3, volume:2, graft_survival:2, mortality:1)

This notebook **re-implements** the trend computation locally (no backend imports) to validate independently.

### References
- SRTR PSR: https://www.srtr.org/reports/program-specific-reports/
- SRTR PSR Technical Methods: https://www.srtr.org/about-the-data/technical-methods-for-the-program-specific-reports/
- TransPlan ADR-022: Historical Trends & Center Trajectories design

In [ ]:
# --- Setup: path configuration, imports, reproducibility ---
import sys, os, json, hashlib
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())

import numpy as np
import pandas as pd
import scipy
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Reproducibility
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# Style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

# Output directory for saved figures
FIGURES_DIR = REPO_ROOT / "notebooks" / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# Load historical trends data and log hash for reproducibility
data_path = REPO_ROOT / "data" / "srtr-historical.json"
with open(data_path) as f:
    hist_data = json.load(f)

file_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()[:12]
print(f"Data file: {data_path.name}")
print(f"SHA-256 (first 12): {file_hash}")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}, SciPy: {scipy.__version__}, Pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}, Seaborn: {sns.__version__}")
print(f"RNG seed: {RNG_SEED}")

# Constants — matching services/trends.py
ORGANS = ["kidney", "liver", "heart", "lung", "pancreas", "intestine"]
ALL_CITIES = [
    "Pittsburgh", "Baltimore", "Philadelphia", "New York", "Minneapolis",
    "Madison", "Chicago", "Cleveland", "St. Louis", "Indianapolis",
    "Omaha", "Rochester", "Nashville", "Durham", "Miami",
    "Dallas", "Houston", "Portland", "Seattle", "San Francisco",
    "Los Angeles", "Palo Alto",
]

# Trend classification thresholds (from services/trends.py)
MIN_POINTS = 3
P_VALUE_THRESHOLD = 0.10
MIN_ANNUAL_CHANGE = {
    "median_wait_months": 0.5,    # 0.5 months/year
    "volume": 3.0,                # 3 transplants/year
    "graft_survival_1yr": 0.2,    # 0.2 percentage points/year
    "patient_survival_1yr": 0.2,
    "mortality_rate": 0.001,      # 0.1 ppt/year
    "delisting_rate": 0.002,
}

# Metrics where lower is better
LOWER_IS_BETTER = {"median_wait_months", "mortality_rate", "delisting_rate"}

# Weighted voting for overall direction
DIRECTION_WEIGHTS = {
    "median_wait_months": 3,
    "volume": 2,
    "graft_survival_1yr": 2,
    "mortality_rate": 1,
}

cities_data = hist_data["cities"]
national_data = hist_data["national"]

print(f"\nLoaded data for {len(cities_data)} cities, {len(ORGANS)} organs")
print(f"Years: {hist_data['_meta']['years']}")
print(f"Source: {hist_data['_meta']['source']}")

In [ ]:
# --- Local re-implementation of trend computation (no backend imports) ---

def compute_metric_trend(years, values, metric_name):
    """
    Compute trend for a single metric time series.
    Mirrors services/trends.py _compute_metric_trend().
    """
    valid_pairs = [(y, v) for y, v in zip(years, values) if v is not None]
    if len(valid_pairs) < MIN_POINTS:
        return {
            "slope": None, "p_value": None, "r_squared": None,
            "direction": "insufficient_data", "change_per_year": None,
            "n_points": len(valid_pairs),
        }
    x = np.array([p[0] for p in valid_pairs], dtype=float)
    y = np.array([p[1] for p in valid_pairs], dtype=float)
    result = stats.linregress(x, y)
    slope = float(result.slope)
    p_value = float(result.pvalue)
    r_squared = float(result.rvalue ** 2)

    min_change = MIN_ANNUAL_CHANGE.get(metric_name, 0.1)
    if p_value > P_VALUE_THRESHOLD or abs(slope) < min_change:
        direction = "stable"
    elif metric_name in LOWER_IS_BETTER:
        direction = "improving" if slope < 0 else "declining"
    else:
        direction = "improving" if slope > 0 else "declining"

    return {
        "slope": round(slope, 6),
        "p_value": round(p_value, 4),
        "r_squared": round(r_squared, 4),
        "direction": direction,
        "change_per_year": round(slope, 4),
        "n_points": len(valid_pairs),
        "intercept": round(float(result.intercept), 4),
        "stderr": round(float(result.stderr), 6),
    }


def compute_overall_direction(metric_trends):
    """
    Weighted voting over metric trends. Mirrors services/trends.py _overall_direction().
    """
    score = 0
    total_weight = 0
    for metric, weight in DIRECTION_WEIGHTS.items():
        t = metric_trends.get(metric)
        if not t or t["direction"] == "insufficient_data":
            continue
        total_weight += weight
        if t["direction"] == "improving":
            score += weight
        elif t["direction"] == "declining":
            score -= weight
    if total_weight == 0:
        return "insufficient_data"
    ratio = score / total_weight
    if ratio > 0.2:
        return "improving"
    elif ratio < -0.2:
        return "declining"
    return "stable"


# Precompute all city/organ trends
TREND_METRICS = ["median_wait_months", "volume", "graft_survival_1yr", "mortality_rate"]
all_trends = {}  # {(city, organ): {metric: trend_dict, ..., "overall": str}}

for city in ALL_CITIES:
    for organ in ORGANS:
        city_organ = cities_data.get(city, {}).get(organ)
        if not city_organ:
            continue
        years = city_organ.get("years", [])
        metric_trends = {}
        for metric in TREND_METRICS:
            values = city_organ.get(metric, [])
            metric_trends[metric] = compute_metric_trend(years, values, metric)
        metric_trends["overall"] = compute_overall_direction(metric_trends)
        all_trends[(city, organ)] = metric_trends

print(f"Computed trends for {len(all_trends)} city-organ combinations")
print(f"Example: Pittsburgh kidney wait_time slope = "
      f"{all_trends[('Pittsburgh', 'kidney')]['median_wait_months']['slope']} months/year")

## 2. Data Overview

Before diving into trend analysis, we examine the data completeness: how many cities have data per organ, the year range, and any gaps (null values) that might affect regression.

In [ ]:
# --- Data Overview: completeness table ---

overview_rows = []
for organ in ORGANS:
    cities_with_data = 0
    total_values = 0
    total_nulls = 0
    year_ranges = []
    for city in ALL_CITIES:
        co = cities_data.get(city, {}).get(organ)
        if co and co.get("years"):
            cities_with_data += 1
            yrs = co["years"]
            year_ranges.append((min(yrs), max(yrs)))
            for metric in TREND_METRICS:
                vals = co.get(metric, [])
                total_values += len(vals)
                total_nulls += sum(1 for v in vals if v is None)
    yr_min = min(r[0] for r in year_ranges) if year_ranges else None
    yr_max = max(r[1] for r in year_ranges) if year_ranges else None
    overview_rows.append({
        "Organ": organ.title(),
        "Cities with Data": cities_with_data,
        "Year Range": f"{yr_min}-{yr_max}" if yr_min else "N/A",
        "Total Metric Values": total_values,
        "Null Values": total_nulls,
        "Completeness (%)": round(100 * (1 - total_nulls / total_values), 1) if total_values > 0 else 0,
    })

overview_df = pd.DataFrame(overview_rows).set_index("Organ")
print("=" * 75)
print("DATA OVERVIEW: SRTR Historical Trends (2019-2025)")
print("=" * 75)
print()
print(overview_df.to_string())
print()
print(f"Total city-organ combinations: {len(all_trends)}")
print(f"Metrics tracked per combination: {len(TREND_METRICS)} "
      f"({', '.join(TREND_METRICS)})")

# National data summary
print("\n--- National Aggregate Data ---")
for organ in ORGANS:
    nat = national_data.get(organ, {})
    yrs = nat.get("years", [])
    n_wait = len([v for v in nat.get("median_wait_months", []) if v is not None])
    n_gs = len([v for v in nat.get("graft_survival_1yr", []) if v is not None])
    print(f"  {organ.title():12s}: {len(yrs)} years, "
          f"{n_wait} wait-time values, {n_gs} graft-survival values")

## 3. National Trends

National aggregate trends for all 6 organs over 2019-2025. Two subplots:
- **Left**: Median wait time (months) — lower is better
- **Right**: 1-year graft survival (%) — higher is better

The 2020 COVID-19 disruption is visible as a wait-time spike / survival dip in several organs.

In [ ]:
# --- National Trends: line plots ---

organ_colors = {
    "kidney": "#1f77b4", "liver": "#ff7f0e", "heart": "#d62728",
    "lung": "#2ca02c", "pancreas": "#9467bd", "intestine": "#8c564b",
}
organ_markers = {
    "kidney": "o", "liver": "s", "heart": "D",
    "lung": "^", "pancreas": "v", "intestine": "P",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Median wait time
ax = axes[0]
for organ in ORGANS:
    nat = national_data.get(organ, {})
    years = nat.get("years", [])
    waits = nat.get("median_wait_months", [])
    if years and waits:
        ax.plot(years, waits, color=organ_colors[organ], marker=organ_markers[organ],
                linewidth=2, markersize=7, label=organ.title())

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Median Wait Time (months)", fontsize=12)
ax.set_title("National Median Wait Time by Organ (2019-2025)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10, framealpha=0.9)
ax.axvspan(2019.5, 2020.5, alpha=0.08, color="red", zorder=0, label=None)
ax.text(2020, ax.get_ylim()[1] * 0.95, "COVID", fontsize=9, ha="center",
        color="red", alpha=0.6, fontstyle="italic")

# Right: Graft survival
ax = axes[1]
for organ in ORGANS:
    nat = national_data.get(organ, {})
    years = nat.get("years", [])
    gs = nat.get("graft_survival_1yr", [])
    if years and gs:
        ax.plot(years, gs, color=organ_colors[organ], marker=organ_markers[organ],
                linewidth=2, markersize=7, label=organ.title())

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("1-Year Graft Survival (%)", fontsize=12)
ax.set_title("National 1-Year Graft Survival by Organ (2019-2025)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10, framealpha=0.9)
ax.axvspan(2019.5, 2020.5, alpha=0.08, color="red", zorder=0)
ax.text(2020, ax.get_ylim()[0] + 0.5, "COVID", fontsize=9, ha="center",
        color="red", alpha=0.6, fontstyle="italic")

fig.suptitle("National Transplant Trends (2019-2025)\n"
             "Red shading = COVID-19 disruption year",
             fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "05-national-trends.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/05-national-trends.png")

# Print summary
print("\n--- National Trend Regressions ---")
for organ in ORGANS:
    nat = national_data.get(organ, {})
    years = nat.get("years", [])
    waits = nat.get("median_wait_months", [])
    gs = nat.get("graft_survival_1yr", [])
    if years and waits:
        wt = compute_metric_trend(years, waits, "median_wait_months")
        gs_info = ""
        if gs:
            gt = compute_metric_trend(years, gs, "graft_survival_1yr")
            gs_info = f", graft slope={gt['change_per_year']:+.2f} pp/yr ({gt['direction']})"
        print(f"  {organ.title():12s}: wait slope={wt['change_per_year']:+.2f} mo/yr "
              f"({wt['direction']}), p={wt['p_value']:.4f}{gs_info}")

## 4. City-Level Trend Heatmap (Kidney)

For kidney (the most common organ and longest waits), we show the **slope** (change per year) in median wait time across all 22 cities as a heatmap. Green = wait time decreasing (improving), red = wait time increasing (declining).

In [ ]:
# --- City-Level Kidney Wait-Time Trend Heatmap ---

kidney_slopes = []
for city in ALL_CITIES:
    t = all_trends.get((city, "kidney"), {}).get("median_wait_months", {})
    slope = t.get("slope")
    direction = t.get("direction", "insufficient_data")
    p_val = t.get("p_value")
    kidney_slopes.append({
        "City": city,
        "Slope (mo/yr)": slope,
        "Direction": direction,
        "p-value": p_val,
    })

slopes_df = pd.DataFrame(kidney_slopes).set_index("City")
slopes_df = slopes_df.sort_values("Slope (mo/yr)", ascending=True)  # Best (most negative) at top

# Build a single-column heatmap
fig, ax = plt.subplots(figsize=(8, 12))

slope_vals = slopes_df["Slope (mo/yr)"].values.reshape(-1, 1)
city_labels = slopes_df.index.tolist()

# Diverging colormap: green for negative slopes (improving), red for positive (declining)
# We reverse RdYlGn so that negative (good) is green
max_abs = max(abs(np.nanmin(slope_vals)), abs(np.nanmax(slope_vals)))
sns.heatmap(
    slope_vals, annot=True, fmt=".2f", cmap="RdYlGn_r",
    center=0, vmin=-max_abs, vmax=max_abs,
    linewidths=0.5, linecolor="white", ax=ax,
    yticklabels=city_labels, xticklabels=["Wait Time Slope\n(months/year)"],
    cbar_kws={"label": "Change in median wait (months/year)", "shrink": 0.6},
)

# Add direction labels on the right
for i, (_, row) in enumerate(slopes_df.iterrows()):
    direction = row["Direction"]
    p_val = row["p-value"]
    symbol = {"improving": "v", "declining": "^", "stable": "-"}.get(direction, "?")
    color = {"improving": "#2ca02c", "declining": "#d62728", "stable": "#666666"}.get(direction, "gray")
    sig = "*" if p_val is not None and p_val < 0.05 else ""
    ax.text(1.15, i + 0.5, f"{symbol} {direction}{sig}", va="center", fontsize=9,
            color=color, fontweight="bold", transform=ax.get_yaxis_transform())

ax.set_title("Kidney: Median Wait-Time Trend by City (2019-2025)\n"
             "(Slope from linear regression; green = improving, red = declining)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "05-kidney-wait-trend-heatmap.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/05-kidney-wait-trend-heatmap.png")

# Print summary table
print("\n--- Kidney Wait-Time Trend Summary ---")
print(slopes_df.to_string())
n_imp = sum(1 for _, r in slopes_df.iterrows() if r["Direction"] == "improving")
n_dec = sum(1 for _, r in slopes_df.iterrows() if r["Direction"] == "declining")
n_stab = sum(1 for _, r in slopes_df.iterrows() if r["Direction"] == "stable")
print(f"\nSummary: {n_imp} improving, {n_stab} stable, {n_dec} declining")

## 5. Trend Direction Distribution

For each organ, how many of the 22 cities are classified as improving, stable, or declining? We show this as a stacked bar chart for both **wait time** and **volume** trends.

In [ ]:
# --- Trend Direction Distribution: stacked bar chart ---

direction_counts = {}
for organ in ORGANS:
    for metric in ["median_wait_months", "volume"]:
        key = f"{organ}_{metric}"
        counts = {"improving": 0, "stable": 0, "declining": 0, "insufficient_data": 0}
        for city in ALL_CITIES:
            t = all_trends.get((city, organ), {}).get(metric, {})
            d = t.get("direction", "insufficient_data")
            counts[d] = counts.get(d, 0) + 1
        direction_counts[key] = counts

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)

dir_colors = {
    "improving": "#2ca02c",
    "stable": "#1f77b4",
    "declining": "#d62728",
    "insufficient_data": "#999999",
}
dir_labels = ["Improving", "Stable", "Declining", "Insufficient Data"]
dir_keys = ["improving", "stable", "declining", "insufficient_data"]

for ax_idx, (metric, metric_label) in enumerate([
    ("median_wait_months", "Wait Time Trend"),
    ("volume", "Volume Trend"),
]):
    ax = axes[ax_idx]
    organ_labels = [o.title() for o in ORGANS]
    x = np.arange(len(ORGANS))
    bottom = np.zeros(len(ORGANS))

    for dk, dl in zip(dir_keys, dir_labels):
        vals = [direction_counts[f"{organ}_{metric}"][dk] for organ in ORGANS]
        bars = ax.bar(x, vals, bottom=bottom, color=dir_colors[dk], label=dl,
                      edgecolor="white", linewidth=0.5, width=0.6)
        # Annotate counts > 0
        for i, v in enumerate(vals):
            if v > 0:
                ax.text(x[i], bottom[i] + v / 2, str(v), ha="center", va="center",
                        fontsize=10, fontweight="bold",
                        color="white" if dk != "insufficient_data" else "black")
        bottom += np.array(vals)

    ax.set_xticks(x)
    ax.set_xticklabels(organ_labels, fontsize=11)
    ax.set_ylabel("Number of Cities" if ax_idx == 0 else "")
    ax.set_title(f"{metric_label} Direction (2019-2025)", fontsize=13, fontweight="bold")
    ax.set_ylim(0, 24)
    if ax_idx == 0:
        ax.legend(fontsize=10, framealpha=0.9, loc="upper right")

fig.suptitle("Trend Direction Distribution by Organ\n"
             "(Based on linear regression: p < 0.10 AND minimum slope threshold)",
             fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "05-trend-direction-distribution.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/05-trend-direction-distribution.png")

# Print counts as table
print("\n--- Direction Counts (Wait Time) ---")
for organ in ORGANS:
    c = direction_counts[f"{organ}_median_wait_months"]
    print(f"  {organ.title():12s}: {c['improving']:2d} improving, "
          f"{c['stable']:2d} stable, {c['declining']:2d} declining, "
          f"{c['insufficient_data']:2d} insufficient")

print("\n--- Direction Counts (Volume) ---")
for organ in ORGANS:
    c = direction_counts[f"{organ}_volume"]
    print(f"  {organ.title():12s}: {c['improving']:2d} improving, "
          f"{c['stable']:2d} stable, {c['declining']:2d} declining, "
          f"{c['insufficient_data']:2d} insufficient")

## 6. Sparkline Gallery (Kidney Wait Times)

Small multiples showing kidney median wait time for all 22 cities (2019-2025), with the national trend as a gray dashed overlay for reference. This is the visual format used in the frontend sparkline charts (M3 modal).

In [ ]:
# --- Sparkline Gallery: kidney wait times, 4x6 grid ---

# National reference for kidney
nat_kidney = national_data.get("kidney", {})
nat_years = nat_kidney.get("years", [])
nat_waits = nat_kidney.get("median_wait_months", [])

# Determine grid layout: 4 columns, enough rows
n_cols = 4
n_rows = int(np.ceil(len(ALL_CITIES) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 2.5), sharex=True)
axes_flat = axes.flatten()

# Sort cities by 2025 wait time (latest year) for visual coherence
city_latest = []
for city in ALL_CITIES:
    co = cities_data.get(city, {}).get("kidney", {})
    waits = co.get("median_wait_months", [])
    latest = waits[-1] if waits and waits[-1] is not None else 999
    city_latest.append((city, latest))
city_latest.sort(key=lambda x: x[1])
sorted_cities = [c[0] for c in city_latest]

for i, city in enumerate(sorted_cities):
    ax = axes_flat[i]
    co = cities_data.get(city, {}).get("kidney", {})
    years = co.get("years", [])
    waits = co.get("median_wait_months", [])

    # Get trend direction for color
    t = all_trends.get((city, "kidney"), {}).get("median_wait_months", {})
    direction = t.get("direction", "stable")
    line_color = {"improving": "#2ca02c", "declining": "#d62728", "stable": "#1f77b4"}.get(
        direction, "#1f77b4")

    # National overlay (gray dashed) — only plot non-None values
    if nat_years and nat_waits:
        nat_valid = [(y, v) for y, v in zip(nat_years, nat_waits) if v is not None]
        if nat_valid:
            ax.plot([p[0] for p in nat_valid], [p[1] for p in nat_valid],
                    color="gray", linestyle="--", linewidth=1.2, alpha=0.5, zorder=1)

    # City sparkline — only plot non-None values
    if years and waits:
        valid = [(y, v) for y, v in zip(years, waits) if v is not None]
        if valid:
            vx = [p[0] for p in valid]
            vy = [p[1] for p in valid]
            ax.plot(vx, vy, color=line_color, linewidth=2, marker="o", markersize=3, zorder=2)

            # Fill area between city and national (only where both have non-None values)
            if nat_years and nat_waits and len(years) == len(nat_years):
                fill_years = []
                fill_city = []
                fill_nat = []
                for y, cv, nv in zip(years, waits, nat_waits):
                    if cv is not None and nv is not None:
                        fill_years.append(y)
                        fill_city.append(cv)
                        fill_nat.append(nv)
                if fill_years:
                    ax.fill_between(fill_years, fill_city, fill_nat,
                                    alpha=0.1, color=line_color)

    # Title with direction badge
    direction_symbol = {"improving": "v", "declining": "^", "stable": "-"}.get(direction, "")
    ax.set_title(f"{city} [{direction_symbol}]", fontsize=10, fontweight="bold",
                 color={"improving": "#2ca02c", "declining": "#d62728"}.get(direction, "#333333"))
    ax.tick_params(axis="both", labelsize=8)
    ax.set_xlim(2018.5, 2025.5)

# Hide unused subplots
for j in range(len(sorted_cities), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle("Kidney Median Wait Time: City Sparklines vs. National Trend (gray dashed)\n"
             "(Sorted by 2025 wait time, ascending; green=improving, red=declining, blue=stable)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "05-kidney-sparklines.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/05-kidney-sparklines.png")

## 7. Regression Quality (Kidney)

Linear regression is a simplification: real trends may be non-linear (especially around the 2020 COVID disruption). We assess goodness-of-fit via R-squared values for the kidney wait-time regression across all 22 cities.

In [ ]:
# --- Regression Quality: R-squared scatter for kidney wait time ---

r2_data = []
for city in ALL_CITIES:
    t = all_trends.get((city, "kidney"), {}).get("median_wait_months", {})
    r2 = t.get("r_squared")
    slope = t.get("slope")
    p_val = t.get("p_value")
    direction = t.get("direction", "insufficient_data")
    if r2 is not None:
        r2_data.append({
            "City": city,
            "R_squared": r2,
            "Slope": slope,
            "p_value": p_val,
            "Direction": direction,
        })

r2_df = pd.DataFrame(r2_data).sort_values("R_squared", ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))

dir_colors_map = {
    "improving": "#2ca02c", "declining": "#d62728",
    "stable": "#1f77b4", "insufficient_data": "#999999",
}
colors_r2 = [dir_colors_map.get(d, "gray") for d in r2_df["Direction"]]

y_pos = np.arange(len(r2_df))
bars = ax.barh(y_pos, r2_df["R_squared"].values, color=colors_r2,
               edgecolor="white", linewidth=0.5, height=0.7)

# Labels
for i, (_, row) in enumerate(r2_df.iterrows()):
    # R2 value on bar
    ax.text(row["R_squared"] + 0.01, i, f"R2={row['R_squared']:.3f}",
            va="center", fontsize=9, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels(r2_df["City"].values, fontsize=10)
ax.set_xlabel("R-squared (goodness of linear fit)", fontsize=12)
ax.set_title("Kidney Wait-Time Linear Regression Quality (R-squared)\n"
             "(Higher = better linear fit; color = trend direction)",
             fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.15)
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.5, linewidth=1)
ax.text(0.5, len(r2_df) - 0.5, "R2=0.50\n(moderate fit)", fontsize=8,
        color="gray", ha="center")
ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend_items = [Patch(facecolor=c, label=d.replace("_", " ").title())
                for d, c in dir_colors_map.items()
                if d in r2_df["Direction"].values]
ax.legend(handles=legend_items, loc="lower right", fontsize=10, framealpha=0.9,
          title="Trend Direction")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "05-regression-quality.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/05-regression-quality.png")

# Summary stats
r2_vals = r2_df["R_squared"].values
print(f"\n--- R-squared Summary (Kidney Wait Time, n={len(r2_vals)} cities) ---")
print(f"  Mean:   {np.mean(r2_vals):.3f}")
print(f"  Median: {np.median(r2_vals):.3f}")
print(f"  Min:    {np.min(r2_vals):.3f} ({r2_df.iloc[-1]['City']})")
print(f"  Max:    {np.max(r2_vals):.3f} ({r2_df.iloc[0]['City']})")
print(f"  Cities with R2 > 0.7: {sum(1 for v in r2_vals if v > 0.7)}")
print(f"  Cities with R2 < 0.3: {sum(1 for v in r2_vals if v < 0.3)}")
print(f"\nNote: Low R2 may indicate non-linear trends (e.g., COVID dip + recovery)")
print(f"or genuinely noisy data at smaller-volume centers.")

## 8. COVID-19 Impact

The 2020 COVID-19 pandemic disrupted transplant operations nationwide: volumes dropped as hospitals paused non-emergency surgeries, wait times spiked, and some outcome metrics deteriorated. This section quantifies the 2020 disruption and recovery pattern.

In [ ]:
# --- COVID Impact: 2020 dip analysis ---

# For each organ, compute national-level 2019 baseline vs 2020 volume change,
# and city-level volume changes

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, organ in enumerate(ORGANS):
    ax = axes[idx // 3, idx % 3]

    # Gather 2019 and 2020 volumes for each city
    baseline_vols = []
    covid_vols = []
    city_labels = []

    for city in ALL_CITIES:
        co = cities_data.get(city, {}).get(organ, {})
        years = co.get("years", [])
        volumes = co.get("volume", [])
        if 2019 in years and 2020 in years:
            idx_2019 = years.index(2019)
            idx_2020 = years.index(2020)
            v2019 = volumes[idx_2019]
            v2020 = volumes[idx_2020]
            if v2019 is not None and v2020 is not None:
                baseline_vols.append(v2019)
                covid_vols.append(v2020)
                city_labels.append(city)

    if baseline_vols:
        # Scatter: 2019 vs 2020 volume
        ax.scatter(baseline_vols, covid_vols, c=organ_colors[organ], s=50,
                   edgecolors="white", linewidth=0.5, zorder=3, alpha=0.8)

        # Identity line (no change)
        lim_max = max(max(baseline_vols), max(covid_vols)) * 1.1
        ax.plot([0, lim_max], [0, lim_max], "k--", linewidth=1, alpha=0.4, label="No change")

        # Compute aggregate change
        total_2019 = sum(baseline_vols)
        total_2020 = sum(covid_vols)
        pct_change = (total_2020 - total_2019) / total_2019 * 100

        # Label cities with biggest drops
        changes = [(c, (v2020 - v2019) / v2019 * 100) for c, v2019, v2020
                    in zip(city_labels, baseline_vols, covid_vols) if v2019 > 0]
        changes.sort(key=lambda x: x[1])
        for c, chg in changes[:2]:  # Label 2 biggest drops
            ci = city_labels.index(c)
            ax.annotate(c, (baseline_vols[ci], covid_vols[ci]), fontsize=7,
                        xytext=(5, -10), textcoords="offset points", alpha=0.7)

        ax.text(0.05, 0.95, f"Aggregate: {pct_change:+.1f}%\n"
                f"n={len(baseline_vols)} cities",
                transform=ax.transAxes, fontsize=9, va="top",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.8))

    ax.set_title(f"{organ.title()}", fontsize=13, fontweight="bold")
    ax.set_xlabel("2019 Volume" if idx >= 3 else "")
    ax.set_ylabel("2020 Volume" if idx % 3 == 0 else "")
    ax.set_aspect("equal", adjustable="box")

fig.suptitle("COVID-19 Impact: 2019 vs. 2020 Transplant Volumes by City\n"
             "(Points below diagonal = volume dropped in 2020)",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "05-covid-impact.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/05-covid-impact.png")

# Recovery analysis: how quickly did volumes return to 2019 levels?
print("\n--- COVID Volume Impact & Recovery ---")
print(f"{'Organ':12s} {'2019 Total':>10s} {'2020 Total':>10s} {'Change':>8s} "
      f"{'Recovery Year':>14s}")
print("-" * 60)
for organ in ORGANS:
    yearly_totals = {}
    for city in ALL_CITIES:
        co = cities_data.get(city, {}).get(organ, {})
        years = co.get("years", [])
        volumes = co.get("volume", [])
        for y, v in zip(years, volumes):
            if v is not None:
                yearly_totals[y] = yearly_totals.get(y, 0) + v

    v2019 = yearly_totals.get(2019, 0)
    v2020 = yearly_totals.get(2020, 0)
    change = (v2020 - v2019) / v2019 * 100 if v2019 > 0 else 0

    # Find first year total >= 2019
    recovery_year = "Not yet"
    for y in range(2021, 2026):
        if yearly_totals.get(y, 0) >= v2019:
            recovery_year = str(y)
            break

    print(f"{organ.title():12s} {v2019:10d} {v2020:10d} {change:+7.1f}% {recovery_year:>14s}")

## 9. Summary

### Key Findings

1. **National trends are broadly positive**: most organ types show decreasing wait times and stable-to-improving graft survival over 2019-2025. The overall trajectory reflects system-wide improvements in donor utilization and surgical technique.

2. **City-level variation is substantial**: within the same organ type, some cities show strong improvement while others are stable or declining. The trend heatmap (Section 4) reveals this geographic heterogeneity clearly.

3. **COVID-19 left a measurable but temporary mark**: 2020 volumes dropped across most organs and cities, with wait times spiking and some survival metrics dipping. Most centers recovered to pre-COVID levels by 2021-2022.

4. **Linear regression is a reasonable but imperfect model**: R-squared values for kidney wait-time trends range widely. Centers with strong monotonic trends have high R-squared; those with COVID disruption or non-linear trajectories show lower fit quality. The linear model captures the dominant signal but may miss acceleration or deceleration.

5. **Pancreas and intestine have sparser data**: smaller program volumes lead to more null values and fewer cities with reportable trends. These organs should be interpreted with extra caution.

### Limitations

| Limitation | Impact | Mitigation |
|-----------|--------|------------|
| Synthetic seed data | Trends are calibrated against published SRTR patterns but are NOT real patient data | Clearly labeled as synthetic; real SRTR PSR data would be used for publication |
| Linear regression only | Non-linear trends (e.g., post-COVID acceleration) may be mischaracterized as stable | R-squared reported; future work could add piecewise or polynomial models |
| 7 data points per series | Short time series limits statistical power (df=5 for regression) | MIN_POINTS=3 threshold; p < 0.10 (not 0.05) to account for low power |
| COVID distortion | 2020 outlier can bias linear slope estimates | Could exclude 2020 for robustness check; not implemented here |
| National aggregates are simple averages | National trends do not weight by center volume | Weighted regression planned for future work |
| No confidence bands on sparklines | Frontend shows point estimates without uncertainty | Backend provides p-values and standard errors for informed interpretation |

### Figures Produced

| Figure | Description |
|--------|-------------|
| `05-national-trends.png` | National median wait time and graft survival line plots (2019-2025) |
| `05-kidney-wait-trend-heatmap.png` | Kidney wait-time slope heatmap across 22 cities |
| `05-trend-direction-distribution.png` | Stacked bar chart of improving/stable/declining counts per organ |
| `05-kidney-sparklines.png` | Small multiples of kidney wait-time sparklines with national overlay |
| `05-regression-quality.png` | R-squared bar chart for kidney wait-time linear model quality |
| `05-covid-impact.png` | 2019 vs 2020 volume scatter plots showing COVID disruption by organ |

### Next Steps

- **Notebook 06** (planned): Policy sensitivity analysis — how allocation policy changes affect wait times and equity across cities.